In [2]:


import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from etils import epath
from flax import struct
from flax.training import orbax_utils
import jax
from jax import numpy as jp
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp
from mujoco_playground.config import locomotion_params
from mujoco_playground import wrapper
from tensorboardX import SummaryWriter
import sys
import json
import time

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

os.environ['CUDA_VISIBLE_DEVICES'] = '1'  # Use only the first GPU (1) for training
# Configure JAX GPU memory settings BEFORE importing jax - OPTIMIZED FOR 40GB A100
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.99'  # Use 98% of GPU memory (~39.2GB out of 40GB)
#os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'  # Use platform allocator
#os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'  # Don't preallocate - grow as needed to avoid fragmentation
#os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Allow dynamic growth


# JAX configuration optimized for large workloads
jax.config.update('jax_enable_x64', False)  # Use float32 to save memory
jax.config.update('jax_traceback_filtering', 'off')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags


ENV_STR = 'Go1JoystickFlatTerrain'

ppo_params = locomotion_params.brax_ppo_config(ENV_STR)



In [4]:

# Specify the directory containing the config.json file
config_dir = "/home/ga53voq/master_thesis/tasks/cave_exploration/logs/cave_exploration-2025-07-15_09-13-47"

# Load configuration from config.json
config_path = os.path.join(config_dir, "config.json")
with open(config_path, 'r') as f:
    config = json.load(f)

# Environment configuration
from tasks.cave_exploration.cave_exploration import default_config as reachbot_config
env_cfg = reachbot_config()
env_cfg.update(config["env_cfg"])
ppo_training_params = dict(ppo_params)
# Overwrite ppo_training_params with the loaded config
ppo_training_params.update(config["ppo_params"])
ppo_training_params["num_timesteps"] = 0
ppo_training_params["num_envs"] = 1

print(f"Loaded configuration from: {config_path}")
print(f"Environment config keys: {list(env_cfg.keys())}")
print(f"PPO training params keys: {list(ppo_training_params.keys())}")

Loaded configuration from: /home/ga53voq/master_thesis/tasks/cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/config.json
Environment config keys: ['Kd_pri', 'Kd_rot', 'Kp_pri', 'Kp_rot', 'action_repeat', 'action_scale', 'cave_batch_size', 'ctrl_dt', 'episode_length', 'history_len', 'lidar_config', 'noise_config', 'pert_config', 'reward_config', 'sim_dt', 'soft_joint_pos_limit_factor', 'stickiness_config']
PPO training params keys: ['action_repeat', 'batch_size', 'discounting', 'entropy_cost', 'episode_length', 'learning_rate', 'max_grad_norm', 'network_factory', 'normalize_observations', 'num_envs', 'num_evals', 'num_minibatches', 'num_resets_per_eval', 'num_timesteps', 'num_updates_per_batch', 'reward_scaling', 'unroll_length']


In [5]:
# Import additional required modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))
from tasks.cave_exploration.cave_exploration import CaveExplore

# Create environment
env = CaveExplore(config=env_cfg)

# Define additional required variables
env_name = "cave_exploration"  # or whatever your environment name is

# Find the latest checkpoint using epath pattern
ckpt_path = config_dir + "/checkpoints"
FINETUNE_PATH = epath.Path(ckpt_path)
latest_ckpts = list(FINETUNE_PATH.glob("*"))
latest_ckpts = [ckpt for ckpt in latest_ckpts if ckpt.is_dir()]
latest_ckpts.sort(key=lambda x: int(x.name))
latest_ckpt = latest_ckpts[-1]
restore_checkpoint_path = latest_ckpt

times = [time.time()]  # for timing measurements

# Network factory setup
network_factory = ppo_networks.make_ppo_networks(observation_size=env.observation_size, action_size=env.action_size)
if "network_factory" in ppo_params:
    if "network_factory" in ppo_training_params:
        del ppo_training_params["network_factory"]
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        **ppo_params.network_factory
    )

Found 3 cave folders in /home/ga53voq/master_thesis/tasks/cave_exploration/environment/caves.
Loading 1 cave environments.
initial_qpos (cave_batch_loader): [ 0.     0.033  0.054  0.994 -0.179  0.    -0.     0.    -0.004 -0.002
  0.001  0.002  0.006 -0.    -0.001 -0.001 -0.001  0.002  0.006]
Floor boxes detected: 5598
Found 4 boom end geoms
Found 5599 floor/wall geoms
CaveExplore task initialized with model: ReachbotModelType.BASIC
CaveExplore task action space: 12
CaveExplore task observation space: {'privileged_state': (164,), 'state': (105,)}


In [7]:
import gc
import jax

gc.collect()

print("Cleared memory and JAX backends")
print(f"Available devices: {jax.devices()}")


# Load the environment configuration from a file

env = CaveExplore(config=env_cfg)

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
)


make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    restore_checkpoint_path=restore_checkpoint_path,  # restore from the checkpoint!
    seed=1,
)

Cleared memory and JAX backends
Available devices: [CudaDevice(id=0)]
Found 3 cave folders in /home/ga53voq/master_thesis/tasks/cave_exploration/environment/caves.
Loading 1 cave environments.
initial_qpos (cave_batch_loader): [ 0.     0.033  0.054  0.994 -0.179  0.    -0.     0.    -0.004 -0.002
  0.001  0.002  0.006 -0.    -0.001 -0.001 -0.001  0.002  0.006]
Floor boxes detected: 4025
Found 4 boom end geoms
Found 4026 floor/wall geoms
CaveExplore task initialized with model: ReachbotModelType.BASIC
CaveExplore task action space: 12
CaveExplore task observation space: {'privileged_state': (164,), 'state': (105,)}


/home/ga53voq/.conda/envs/pyenv/lib/python3.12/site-packages/jax/_src/interpreters/xla.py:119: RuntimeWarning: overflow encountered in cast
  return np.asarray(x, dtypes.canonicalize_dtype(x.dtype))


# Video Rollout

Generate videos from the trained model to visualize the learned behavior.

In [ ]:
# Video creation from trained model
import imageio
import gc
from datetime import datetime

# Free up training memory before rendering
gc.collect()

# Setup JIT compiled functions for inference
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
inference_fn = make_inference_fn(params, deterministic=True)
jit_inference_fn = jax.jit(inference_fn)

print("Setting up rollout for video creation...")

# Rollout parameters
rng = jax.random.PRNGKey(0)
n_episodes = 5
rollout_steps = 10000

# Set command (if needed for environment)
x_vel = 0.2
y_vel = 0.2
yaw_vel = 0.0
command = jp.array([x_vel, y_vel, yaw_vel])

# Create video output directory in the same config directory
video_dir = os.path.join(config_dir, "videos")
os.makedirs(video_dir, exist_ok=True)

# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {rollout_steps} steps each...")
episode_rewards = []

for episode in range(n_episodes):
    print(f"Episode {episode + 1}/{n_episodes}")
    state = jit_reset(rng)
    rollout = [state]  # Reset rollout for each episode
    episode_reward = 0.0
    
    for i in range(rollout_steps):
        if i % 500 == 0:
            print(f"  Step {i}/{rollout_steps}, Current reward: {episode_reward:.3f}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)
        
        # Accumulate reward for this episode
        episode_reward += float(state.reward)
        
        if state.done:
            print(f"Episode {episode + 1} ended at step {i} with reward: {episode_reward:.3f}")
            break
            
        rollout.append(state)

    episode_rewards.append(episode_reward)
    print(f"Episode {episode + 1} completed with {len(rollout)} states and total reward: {episode_reward:.3f}")

    # Render video
    print("Rendering video...")
    render_every = 1  # Render every frame
    width = 1920      # Full HD width
    height = 1080     # Full HD height

    frames = env.render(rollout[::render_every], camera='track_global', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(video_dir, f'continued_training_episode_{episode}_reward_{episode_reward:.1f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully: Episode {episode + 1}, Reward: {episode_reward:.3f}")

# Print summary of all episodes
print("\n=== EPISODE REWARD SUMMARY ===")
for i, reward in enumerate(episode_rewards):
    print(f"Episode {i + 1}: {reward:.3f}")
print(f"Average reward: {sum(episode_rewards)/len(episode_rewards):.3f}")
print(f"Best episode: {episode_rewards.index(max(episode_rewards)) + 1} with reward {max(episode_rewards):.3f}")
print(f"Worst episode: {episode_rewards.index(min(episode_rewards)) + 1} with reward {min(episode_rewards):.3f}")

print("\n=== VIDEO GENERATION COMPLETE ===")
print(f"Videos saved to: {video_dir}")
print(f"Episode summary: Average reward {sum(episode_rewards)/len(episode_rewards):.1f}, Best: {max(episode_rewards):.1f}, Worst: {min(episode_rewards):.1f}")

Setting up rollout for video creation...
Running rollout for 5 episode(s) with 10000 steps each...
Episode 1/5


2025-07-15 17:18:27.623448: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


  Step 0/10000, Current reward: 0.000
Max contacts updated to 256 based on current step.
  Step 500/10000, Current reward: -0.902
  Step 1000/10000, Current reward: -1.884
Episode 1 ended at step 1251 with reward: -2.385
Episode 1 completed with 1252 states and total reward: -2.385
Rendering video...


100%|██████████| 1252/1252 [00:27<00:00, 45.21it/s]
/home/ga53voq/.conda/envs/pyenv/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(


Rendered 1252 frames
Saving video to /home/ga53voq/master_thesis/tasks/cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/videos/continued_training_episode_0_reward_-2.4.mp4 at 50.0 FPS...
Video saved successfully: Episode 1, Reward: -2.385
Episode 2/5
  Step 0/10000, Current reward: 0.000
Episode 2 ended at step 78 with reward: -0.115
Episode 2 completed with 79 states and total reward: -0.115
Rendering video...


...

100%|██████████| 635/635 [00:18<00:00, 33.75it/s]


Rendered 635 frames
Saving video to /home/ga53voq/master_thesis/tasks/cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/videos/continued_training_episode_3_reward_-1.2.mp4 at 50.0 FPS...
Video saved successfully: Episode 4, Reward: -1.177
Episode 5/5
  Step 0/10000, Current reward: 0.000
  Step 500/10000, Current reward: -0.950
  Step 1000/10000, Current reward: -1.950
Episode 5 ended at step 1251 with reward: -2.455
Episode 5 completed with 1252 states and total reward: -2.455
Rendering video...


100%|██████████| 1252/1252 [00:36<00:00, 34.09it/s]


Rendered 1252 frames
Saving video to /home/ga53voq/master_thesis/tasks/cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/videos/continued_training_episode_4_reward_-2.5.mp4 at 50.0 FPS...
Video saved successfully: Episode 5, Reward: -2.455

=== EPISODE REWARD SUMMARY ===
Episode 1: -2.385
Episode 2: -0.115
Episode 3: -2.566
Episode 4: -1.177
Episode 5: -2.455
Average reward: -1.740
Best episode: 2 with reward -0.115
Worst episode: 3 with reward -2.566

=== VIDEO GENERATION COMPLETE ===
Videos saved to: /home/ga53voq/master_thesis/tasks/cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/videos
Episode summary: Average reward -1.7, Best: -0.1, Worst: -2.6


: 